# E3, bloque B: exportación a ONNX de las 10 redes del E2

1. Entorno de ejecución → Cambiar tipo → **GPU** (sirve una T4; no hace falta la L4).
2. Ejecutar las celdas en orden. Lee los checkpoints de `/content/drive/MyDrive/tesis_imo_e2_arquitecturas_reales_v2` (solo lectura) y escribe en
   `/content/drive/MyDrive/tesis_imo_e3_artefactos_v1` una carpeta por red con el `.onnx` y su `metadata.json`, más `resumen_exportacion.json`.
3. Al terminar, descargar la carpeta `tesis_imo_e3_artefactos_v1` completa desde Drive.

No se usa test. Una red que no exporta no detiene a las demás: queda con su error en `metadata.json`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, os, json, hashlib, zipfile
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['mountpoint', '-q', '/content/drive'], check=True)
CORRIDAS = Path('/content/drive/MyDrive/tesis_imo_e2_arquitecturas_reales_v2')
assert CORRIDAS.is_dir(), 'No está la carpeta de corridas del E2 en Drive'
OUTPUT = Path('/content/drive/MyDrive/tesis_imo_e3_artefactos_v1')
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
EXPECTED_ZIP_SHA256 = '954f4fe5663d9fedd676dcee4f1f7fb9086a2a7f5c1907ce902b5df4c109cb49'
ZIP = next((p for p in [Path('/content/paquete_colab_e2_v2.zip'), Path('/content/drive/MyDrive/paquete_colab_e2_v2.zip')] if p.exists()), None)
if ZIP is None: raise FileNotFoundError('Sube paquete_colab_e2_v2.zip al panel de archivos y repite esta celda.')
def sha_file(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()
assert sha_file(ZIP)==EXPECTED_ZIP_SHA256, 'ZIP incorrecto o modificado; usa el archivo v2 entregado con este notebook'
BASE=Path('/content/imo_e2_v2')
BASE.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    for n in z.namelist():
        if not (BASE/n).resolve().is_relative_to(BASE): raise ValueError('Ruta no válida en ZIP')
    z.extractall(BASE)
manifest=json.loads((BASE/'PACKAGE_MANIFEST.json').read_text())
for name, expected in manifest['files'].items():
    assert sha_file(BASE/name)==expected, name
PROJECT=BASE/'tesis_imo_para_codex'
print('Paquete íntegro:',len(manifest['files']),'archivos')


## Entorno aislado y fijado
Se crea Python 3.11 fuera del kernel de Colab. La instalación puede tardar varios minutos; no cambia el Python del notebook. No se compilan extensiones CUDA: se usan ruedas publicadas por sus autores para torch 2.5/CUDA 12.


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','uv==0.6.9'],check=True)
ENV=Path('/content/imo-e2-env')
if not (ENV/'bin/python').exists():
    subprocess.run([sys.executable,'-m','uv','venv','--python','3.11',str(ENV)],check=True)
PY=str(ENV/'bin/python')
def pip_install(*items):
    subprocess.run([sys.executable,'-m','uv','pip','install','--python',PY,*items],check=True)
pip_install('torch==2.5.1','torchvision==0.20.1','--index-url','https://download.pytorch.org/whl/cu121')
pip_install('-r',str(PROJECT/'requirements-e2.txt'))
pip_install('transformers==4.46.3')
pip_install('--no-deps','https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/causal_conv1d-1.5.0.post8%2Bcu12torch2.5cxx11abiFALSE-cp311-cp311-linux_x86_64.whl')
pip_install('--no-deps','https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4%2Bcu12torch2.5cxx11abiFALSE-cp311-cp311-linux_x86_64.whl')
subprocess.run([PY,'-c','import torch, torchvision, mamba_ssm; assert torch.cuda.is_available(); print(torch.__version__,torch.cuda.get_device_name(0))'],check=True)
freeze=subprocess.check_output([sys.executable,'-m','uv','pip','freeze','--python',PY],text=True)
(OUTPUT/'environment.freeze.txt').write_text(freeze)


In [ ]:
pip_install('onnx==1.17.0', 'onnxruntime==1.20.1', 'fvcore==0.1.5.post20221221')
subprocess.run([PY, '-c', 'import onnx, onnxruntime; print(onnx.__version__, onnxruntime.__version__)'], check=True)


In [ ]:
SCRIPT = Path('/content/exportar_onnx.py')
SCRIPT.write_text('"""Bloque B del E3: elige el checkpoint de cada red, lo exporta a ONNX y hace la prueba de ida y vuelta.\n\nCorre en Colab, en el entorno del benchmark del E2 (el `PY` del cuaderno, con torch 2.5.1 y mamba_ssm), con el\npaquete del E2 descomprimido (`--e2` = carpeta tesis_imo_para_codex) y los checkpoints en Drive.\n\nReglas, fijadas antes de mirar resultados del E3:\n- **Configuración:** cada red tuvo una sola receta en el E2 (config/experimento_e2.json), así que la mejor\n  configuración es esa. Si la búsqueda de arquitectura de las compactas (B14) elige otra, sus artefactos se\n  exportan después con la arquitectura elegida (pendiente; no lo cubre esta versión).\n- **Reentrenamiento final (p. 6):** la política, idéntica para todas las redes y para el original, es que el modelo\n  final es el checkpoint de la época elegida en validación, entrenado sobre el conjunto de entrenamiento completo\n  (las 1.015 imágenes de train). No hay un segundo entrenamiento: la validación solo eligió la época.\n- **Artefacto por red:** la semilla cuyo Dice medio por grupo en validación es la mediana de las semillas\n  disponibles (con 5 semillas, la tercera; empate: la semilla menor). Representa a la red sin elegir su mejor corrida.\n- **Prueba de ida y vuelta:** en las 218 imágenes de validación, error absoluto máximo de la probabilidad < 1e-4,\n  coincidencia de la máscara (umbral 0,5) y diferencia del Dice medio por grupo entre el ONNX y el modelo en memoria.\n- **Tamaño:** si el ONNX pasa de 95 MB (límite de GitHub, 100 MB), se cuantiza a INT8 dinámico y se mide la pérdida de\n  Dice en validación; se acepta si es menor que δ = 0,02. Si no se acepta o la exportación falla, la red queda con\n  resultados precalculados en el dashboard y el motivo en `metadata.json`.\n- **Costo:** latencia por imagen en CPU con ONNX Runtime (mediana y p95 de 50 pasadas tras 5 de calentamiento, un hilo\n  y todos los hilos), parámetros y FLOPs (fvcore, si está), memoria residente pico durante la inferencia.\n\nUso:\n  python exportar_onnx.py --e2 /content/imo_e2_v2/tesis_imo_para_codex \\\n      --corridas /content/drive/MyDrive/tesis_imo_e2_arquitecturas_reales_v2 --salida /content/drive/MyDrive/tesis_imo_e3_artefactos_v1 \\\n      [--modelos fcn unet ...] [--dispositivo cuda]\n"""\nimport argparse\nimport hashlib\nimport json\nimport os\nimport platform\nimport statistics\nimport sys\nimport time\nimport traceback\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\n\nTOL_PROB = 1e-4\nLIMITE_MB = 95\nDELTA = 0.02\nOPSET = 17\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for b in iter(lambda: f.read(1 << 20), b\'\'):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef dice_por_grupo(probs, filas, metrics):\n    grupos = defaultdict(list)\n    for p, (fila, y, v) in zip(probs, filas):\n        grupos[fila[\'group_id\']].append(metrics(p, y, v)[\'Dice\'])\n    return float(np.mean([np.mean(x) for x in grupos.values()]))\n\n\ndef elegir_semilla(corridas, modelo):\n    """Semilla de Dice de validación mediano entre las corridas completas del modelo."""\n    res = []\n    for r in sorted(Path(corridas).glob(f\'{modelo}_seed*/result.json\')):\n        d = json.loads(r.read_text())\n        if d.get(\'completed\'):\n            res.append((d[\'best_val_group_Dice\'], d[\'seed\'], r.parent / d[\'best_file\']))\n    if not res:\n        raise FileNotFoundError(f\'Sin corridas completas de {modelo} en {corridas}\')\n    res.sort(key=lambda t: (t[0], t[1]))\n    mediana = res[(len(res) - 1) // 2]\n    return {\'semillas_disponibles\': [s for _, s, _ in sorted(res, key=lambda t: t[1])],\n            \'val_por_semilla\': {s: v for v, s, _ in res}, \'semilla\': mediana[1], \'val_dice\': mediana[0],\n            \'checkpoint\': mediana[2]}\n\n\ndef latencia(sesion, x, hilos):\n    import onnxruntime as ort\n    nombre = sesion.get_inputs()[0].name\n    for _ in range(5):\n        sesion.run(None, {nombre: x})\n    t = []\n    for _ in range(50):\n        t0 = time.perf_counter()\n        sesion.run(None, {nombre: x})\n        t.append((time.perf_counter() - t0) * 1000)\n    t.sort()\n    return {\'hilos\': hilos, \'mediana_ms\': statistics.median(t), \'p95_ms\': t[int(0.95 * len(t)) - 1]}\n\n\ndef sesion_cpu(ruta, hilos=None):\n    import onnxruntime as ort\n    op = ort.SessionOptions()\n    if hilos:\n        op.intra_op_num_threads = hilos\n    return ort.InferenceSession(str(ruta), op, providers=[\'CPUExecutionProvider\'])\n\n\ndef exportar_uno(nombre, args, MODELS, main_logits, filas_val, metrics, meta):\n    import psutil\n    import torch\n    from torch import nn\n\n    destino = Path(args.salida) / nombre\n    destino.mkdir(parents=True, exist_ok=True)\n    meta.update({\'modelo\': nombre, \'fecha\': time.strftime(\'%Y-%m-%d %H:%M:%S\'), \'estado\': \'iniciado\'})\n    eleccion = elegir_semilla(args.corridas, nombre)\n    meta.update({k: (str(v) if k == \'checkpoint\' else v) for k, v in eleccion.items()})\n    meta[\'checkpoint_sha256\'] = sha(eleccion[\'checkpoint\'])\n\n    dispositivo = torch.device(args.dispositivo)\n    modelo = MODELS[nombre]()\n    estado = torch.load(eleccion[\'checkpoint\'], map_location=\'cpu\', weights_only=False)\n    modelo.load_state_dict(estado[\'model\'])\n    modelo.eval().to(dispositivo)\n\n    class Probabilidad(nn.Module):\n        """Salida única del artefacto: probabilidad por píxel, sigmoid del logit principal."""\n        def __init__(self, m):\n            super().__init__()\n            self.m = m\n\n        def forward(self, x):\n            return torch.sigmoid(main_logits(self.m(x)))\n\n    envuelto = Probabilidad(modelo).eval()\n    meta[\'parametros\'] = int(sum(p.numel() for p in modelo.parameters()))\n    x0 = torch.from_numpy(filas_val[0][0][None]).to(dispositivo)\n\n    try:\n        from fvcore.nn import FlopCountAnalysis\n        meta[\'flops\'] = int(FlopCountAnalysis(envuelto, x0).unsupported_ops_warnings(False).total())\n        meta[\'flops_fuente\'] = \'fvcore (multiplicación-suma contada como 1)\'\n    except Exception as e:  # noqa: BLE001\n        meta[\'flops\'], meta[\'flops_fuente\'] = None, f\'no medido: {type(e).__name__}\'\n\n    # Salidas del modelo en memoria, imagen por imagen, con el mismo orden que el ONNX.\n    with torch.no_grad():\n        p_mem = [envuelto(torch.from_numpy(x[None]).to(dispositivo))[0, 0].float().cpu().numpy()\n                 for x, _, _ in filas_val_xyv(filas_val)]\n    meta[\'val_dice_memoria\'] = dice_por_grupo(p_mem, filas_meta(filas_val), metrics)\n\n    ruta = destino / f\'{nombre}.onnx\'\n    envuelto_cpu = envuelto.to(\'cpu\')\n    torch.onnx.export(envuelto_cpu, x0.cpu(), str(ruta), opset_version=OPSET, input_names=[\'imagen\'],\n                      output_names=[\'probabilidad\'], do_constant_folding=True, dynamo=False)\n    import onnx\n    onnx.checker.check_model(str(ruta))\n    meta[\'onnx_mb\'] = ruta.stat().st_size / 2 ** 20\n\n    def evaluar(ruta_onnx):\n        s = sesion_cpu(ruta_onnx)\n        nombre_in = s.get_inputs()[0].name\n        proc = psutil.Process()\n        rss0, pico = proc.memory_info().rss, 0\n        p = []\n        for x, _, _ in filas_val_xyv(filas_val):\n            p.append(s.run(None, {nombre_in: x[None]})[0][0, 0])\n            pico = max(pico, proc.memory_info().rss)\n        return p, (pico - rss0) / 2 ** 20, s\n\n    p_onnx, mem_mb, s = evaluar(ruta)\n    dif = max(float(np.abs(a - b).max()) for a, b in zip(p_mem, p_onnx))\n    coinc = float(np.mean([((a >= .5) == (b >= .5)).mean() for a, b in zip(p_mem, p_onnx)]))\n    meta[\'ida_y_vuelta\'] = {\'imagenes\': len(p_onnx), \'error_abs_max\': dif, \'tolerancia\': TOL_PROB,\n                            \'coincidencia_mascara\': coinc, \'val_dice_onnx\': dice_por_grupo(p_onnx, filas_meta(filas_val), metrics),\n                            \'aprueba\': dif < TOL_PROB}\n    meta[\'ida_y_vuelta\'][\'diferencia_dice\'] = meta[\'ida_y_vuelta\'][\'val_dice_onnx\'] - meta[\'val_dice_memoria\']\n    meta[\'memoria_inferencia_mb\'] = mem_mb\n    x_np = filas_val[0][0][None]\n    meta[\'latencia_cpu\'] = [latencia(sesion_cpu(ruta, 1), x_np, 1), latencia(s, x_np, os.cpu_count())]\n    meta[\'artefacto\'] = ruta.name\n\n    if meta[\'onnx_mb\'] > LIMITE_MB:\n        from onnxruntime.quantization import QuantType, quantize_dynamic\n        ruta_q = destino / f\'{nombre}_int8.onnx\'\n        quantize_dynamic(str(ruta), str(ruta_q), weight_type=QuantType.QInt8)\n        p_q, _, sq = evaluar(ruta_q)\n        dq = dice_por_grupo(p_q, filas_meta(filas_val), metrics) - meta[\'val_dice_memoria\']\n        meta[\'int8\'] = {\'mb\': ruta_q.stat().st_size / 2 ** 20, \'diferencia_dice_val\': dq,\n                        \'acepta\': abs(dq) < DELTA and ruta_q.stat().st_size / 2 ** 20 <= LIMITE_MB,\n                        \'latencia_cpu\': latencia(sq, x_np, os.cpu_count())}\n        if meta[\'int8\'][\'acepta\']:\n            meta[\'artefacto\'] = ruta_q.name\n        else:\n            meta[\'artefacto\'] = None\n            meta[\'motivo_sin_artefacto\'] = \'No cabe en GitHub ni cuantizado sin perder δ: resultados precalculados\'\n    meta[\'artefacto_sha256\'] = sha(destino / meta[\'artefacto\']) if meta[\'artefacto\'] else None\n    meta[\'preprocesamiento\'] = {\'tamano\': 256, \'letterbox\': True, \'normalizacion\': args.norm,\n                                \'entrada\': \'float32 [1, 3, 256, 256], RGB normalizado; padding = 0\',\n                                \'salida\': \'float32 [1, 1, 256, 256], probabilidad; máscara = probabilidad >= 0,5\'}\n    meta[\'entorno\'] = {\'python\': platform.python_version(), \'torch\': torch.__version__,\n                       \'onnx\': __import__(\'onnx\').__version__, \'onnxruntime\': __import__(\'onnxruntime\').__version__,\n                       \'opset\': OPSET, \'cpu\': platform.processor() or platform.machine(), \'hilos\': os.cpu_count()}\n    meta[\'estado\'] = \'ok\' if meta[\'ida_y_vuelta\'][\'aprueba\'] else \'ida_y_vuelta_fallida\'\n\n\n\ndef filas_val_xyv(filas):\n    return [(x, y, v) for x, y, v, _ in filas]\n\n\ndef filas_meta(filas):\n    return [(f, y, v) for _, y, v, f in filas]\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--e2\', required=True)\n    ap.add_argument(\'--corridas\', required=True)\n    ap.add_argument(\'--salida\', required=True)\n    ap.add_argument(\'--modelos\', nargs=\'+\')\n    ap.add_argument(\'--dispositivo\', default=\'cpu\')\n    args = ap.parse_args()\n    proyecto = Path(args.e2)\n    sys.path.insert(0, str(proyecto / \'scripts\'))\n    from dataset_ready import load_example, metrics, partition\n    from modelos_e2 import MODELS_E2, main_logits\n    args.norm = json.loads((proyecto / \'resultados/preentrenamiento/normalization_train.json\').read_text())\n    filas = []\n    for fila in sorted(partition(\'val\'), key=lambda r: r[\'id\']):\n        x, y, v = load_example(fila)\n        filas.append((x, y, v, fila))\n    salida = Path(args.salida)\n    salida.mkdir(parents=True, exist_ok=True)\n    resumen = {}\n    for nombre in args.modelos or list(MODELS_E2):\n        print(\'==\', nombre, flush=True)\n        meta = {}\n        try:\n            exportar_uno(nombre, args, MODELS_E2, main_logits, filas, metrics, meta)\n        except Exception as e:  # noqa: BLE001  una red que no exporta no detiene a las demás\n            # Se conserva lo ya medido (semilla, parámetros, Dice en memoria) y se agrega el error.\n            meta.update({\'estado\': \'fallo\', \'error\': f\'{type(e).__name__}: {e}\', \'traza\': traceback.format_exc()[-2000:],\n                         \'motivo_sin_artefacto\': \'Exportación fallida: resultados precalculados\'})\n        (salida / nombre).mkdir(exist_ok=True)\n        (salida / nombre / \'metadata.json\').write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding=\'utf-8\')\n        resumen[nombre] = {k: meta.get(k) for k in [\'estado\', \'semilla\', \'onnx_mb\', \'artefacto\', \'error\']}\n        resumen[nombre][\'ida_y_vuelta\'] = (meta.get(\'ida_y_vuelta\') or {}).get(\'error_abs_max\')\n        print(json.dumps(resumen[nombre], ensure_ascii=False, default=str), flush=True)\n    (salida / \'resumen_exportacion.json\').write_text(json.dumps(resumen, ensure_ascii=False, indent=2, default=str), encoding=\'utf-8\')\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')
assert hashlib.sha256(SCRIPT.read_bytes()).hexdigest() == '9b1fcbb84a2e3ea6df9f9ed6513b440a98c8633f35bd4f06d970fec44d6cd7a3', 'Script alterado'


## Exportar
Tarda de 20 a 60 minutos. Si se cae la sesión, repetir todas las celdas: las redes ya exportadas se vuelven a exportar (no hay reanudación parcial).


In [ ]:
subprocess.run([PY, str(SCRIPT), '--e2', str(PROJECT), '--corridas', str(CORRIDAS),
                '--salida', str(OUTPUT), '--dispositivo', 'cuda'], check=True)
print((OUTPUT / 'resumen_exportacion.json').read_text())
